In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import os
os.chdir('/content/drive/MyDrive')

!mkdir -p olist-customer-intelligence/{data/raw,data/processed,notebooks,sql,src,models,reports,app/pages,docs}
os.chdir('/content/drive/MyDrive/M5-forecasting')
!touch src/__init__.py
print("Now in:", os.getcwd())

Now in: /content/drive/MyDrive/M5-forecasting


In [19]:
%%writefile src/inventory_optimizer.py
"""Newsvendor inventory optimization from probabilistic forecasts."""
import numpy as np
import pandas as pd
from scipy import stats


def critical_ratio(unit_margin, unit_holding_cost):
    """CR = Cu / (Cu + Co). The optimal service level (fractile)."""
    return unit_margin / (unit_margin + unit_holding_cost)


def optimal_order_from_quantiles(p10, p50, p90, cr):
    """Interpolate the optimal order quantity from three known quantiles.
    Fits a normal via P50 (mean) and (P90-P10)/(2*1.2816) (sigma), then
    reads the CR-th quantile. Robust when only 3 quantiles are available."""
    sigma = (p90 - p10) / (2 * 1.2816)          # 1.2816 = z(0.9)
    sigma = np.maximum(sigma, 1e-6)
    z = stats.norm.ppf(np.clip(cr, 1e-4, 1 - 1e-4))
    return np.maximum(p50 + z * sigma, 0)


def newsvendor_cost(order_qty, actual_demand, unit_margin, unit_holding_cost):
    """Realized cost: lost margin on stockouts + holding cost on overage."""
    understock = np.maximum(actual_demand - order_qty, 0) * unit_margin
    overstock = np.maximum(order_qty - actual_demand, 0) * unit_holding_cost
    return understock + overstock


def compare_policies(fc_df, unit_margin, unit_holding_cost,
                     truth_col="sales"):
    """Compare ordering at P50 vs newsvendor-optimal against realized cost.
    fc_df: [id, date, p10, p50, p90, sales]. Returns summary dict."""
    cr = critical_ratio(unit_margin, unit_holding_cost)
    opt_order = optimal_order_from_quantiles(
        fc_df["p10"].values, fc_df["p50"].values, fc_df["p90"].values, cr)

    cost_p50 = newsvendor_cost(fc_df["p50"].values, fc_df[truth_col].values,
                               unit_margin, unit_holding_cost)
    cost_opt = newsvendor_cost(opt_order, fc_df[truth_col].values,
                               unit_margin, unit_holding_cost)
    return {
        "critical_ratio": round(cr, 3),
        "total_cost_p50": float(cost_p50.sum()),
        "total_cost_newsvendor": float(cost_opt.sum()),
        "savings": float(cost_p50.sum() - cost_opt.sum()),
        "savings_pct": round(100 * (cost_p50.sum() - cost_opt.sum())
                             / max(cost_p50.sum(), 1e-9), 2),
    }

Overwriting src/inventory_optimizer.py


In [20]:
import pandas as pd, numpy as np
from src.inventory_optimizer import (critical_ratio, optimal_order_from_quantiles,
                                     newsvendor_cost, compare_policies)

lgb_fc = pd.read_parquet("data/processed/forecasts_lightgbm.parquet")

# attach a representative price per item (last known sell_price)
prices = pd.read_parquet("data/processed/features.parquet",
                         columns=["id","date","sell_price"])
last_price = (prices.sort_values("date").groupby("id", observed=True)["sell_price"]
              .last().rename("price"))
lgb_fc = lgb_fc.merge(last_price, on="id", how="left")
lgb_fc["price"] = lgb_fc["price"].fillna(lgb_fc["price"].median())

# economics: 30% margin, holding cost 15% of price per unit per period
lgb_fc["unit_margin"]  = lgb_fc["price"] * 0.30
lgb_fc["unit_holding"] = lgb_fc["price"] * 0.15
print(lgb_fc[["id","p10","p50","p90","sales","price"]].head())

                            id  p10       p50       p90  sales  price
0  FOODS_1_001_CA_1_evaluation  0.0  0.996431  2.814275    2.0   2.24
1  FOODS_1_001_CA_1_evaluation  0.0  0.996243  2.530950    0.0   2.24
2  FOODS_1_001_CA_1_evaluation  0.0  0.992066  2.485720    0.0   2.24
3  FOODS_1_001_CA_1_evaluation  0.0  0.704538  2.434704    0.0   2.24
4  FOODS_1_001_CA_1_evaluation  0.0  0.705552  2.712722    0.0   2.24


In [21]:
# per-row optimal order using each item's own critical ratio
cr = critical_ratio(lgb_fc["unit_margin"].values, lgb_fc["unit_holding"].values)
opt_order = optimal_order_from_quantiles(
    lgb_fc["p10"].values, lgb_fc["p50"].values, lgb_fc["p90"].values, cr)

cost_p50 = newsvendor_cost(lgb_fc["p50"].values, lgb_fc["sales"].values,
                           lgb_fc["unit_margin"].values, lgb_fc["unit_holding"].values)
cost_opt = newsvendor_cost(opt_order, lgb_fc["sales"].values,
                           lgb_fc["unit_margin"].values, lgb_fc["unit_holding"].values)

print(f"Mean critical ratio: {cr.mean():.3f}")
print(f"Total cost — order at P50:        ${cost_p50.sum():,.0f}")
print(f"Total cost — newsvendor optimal:  ${cost_opt.sum():,.0f}")
print(f"Savings: ${cost_p50.sum()-cost_opt.sum():,.0f} "
      f"({100*(cost_p50.sum()-cost_opt.sum())/cost_p50.sum():.1f}%)")

Mean critical ratio: 0.667
Total cost — order at P50:        $684,899
Total cost — newsvendor optimal:  $656,976
Savings: $27,923 (4.1%)


In [22]:
rows = []
for margin_pct, hold_pct in [(0.20,0.20),(0.30,0.15),(0.50,0.10),(0.70,0.05)]:
    um = lgb_fc["price"].values * margin_pct
    uh = lgb_fc["price"].values * hold_pct
    crx = critical_ratio(um, uh)
    oq = optimal_order_from_quantiles(lgb_fc["p10"].values, lgb_fc["p50"].values,
                                      lgb_fc["p90"].values, crx)
    c50 = newsvendor_cost(lgb_fc["p50"].values, lgb_fc["sales"].values, um, uh).sum()
    copt = newsvendor_cost(oq, lgb_fc["sales"].values, um, uh).sum()
    rows.append({"margin":f"{int(margin_pct*100)}%", "hold":f"{int(hold_pct*100)}%",
                 "crit_ratio":round(crx.mean(),3),
                 "savings_pct":round(100*(c50-copt)/c50,1)})
sens = pd.DataFrame(rows)
print(sens.to_string(index=False))

margin hold  crit_ratio  savings_pct
   20%  20%       0.500          0.0
   30%  15%       0.667          4.1
   50%  10%       0.833         29.2
   70%   5%       0.933         55.5


In [23]:
biz = {
    "mean_critical_ratio": float(cr.mean()),
    "cost_p50": float(cost_p50.sum()),
    "cost_newsvendor": float(cost_opt.sum()),
    "savings": float(cost_p50.sum()-cost_opt.sum()),
    "savings_pct": float(100*(cost_p50.sum()-cost_opt.sum())/cost_p50.sum()),
}
pd.DataFrame([biz]).to_csv("data/processed/business_results.csv", index=False)
sens.to_csv("data/processed/sensitivity_analysis.csv", index=False)
print("saved business_results.csv + sensitivity_analysis.csv")
print(biz)

saved business_results.csv + sensitivity_analysis.csv
{'mean_critical_ratio': 0.6666669249534607, 'cost_p50': 684898.9283360839, 'cost_newsvendor': 656975.9750947907, 'savings': 27922.953241293202, 'savings_pct': 4.076945091610838}


##Newsvendor Inventory Optimization

**The business layer: turning probabilistic forecasts into dollar decisions.**

**Method — newsvendor model:**
- Optimal order = demand quantile at the **critical ratio** CR = Cu / (Cu + Co)
- Cu = unit margin (cost of understocking), Co = unit holding cost (cost of overstocking)
- P10/P50/P90 quantiles → fit normal (P50 as mean, (P90−P10)/2z₀.₉ as σ) → read the CR-th fractile
- High margin → order above median; low margin/perishable → order below. Only a quantile model enables this.

**Economics (M5 has no cost data → representative assumptions):**
- Unit margin = 30% of sell_price; holding cost = 15% of price/period (base case)

**Headline result (28-day test, all leaves):**
- Order-at-P50 (point forecast) total cost: **$684,899**
- Newsvendor-optimal (quantile-aware) total cost: **$656,976**
- **Savings: $27,923 (4.1%)** at base-case economics

**Sensitivity — the key insight:**

| Margin | Hold | Crit. ratio | Savings |
|--------|------|-------------|---------|
| 20% | 20% | 0.500 | 0.0% |
| 30% | 15% | 0.667 | 4.1% |
| 50% | 10% | 0.833 | 29.2% |
| 70% | 5% | 0.933 | 55.5% |

- At CR = 0.5 (symmetric costs), optimal = P50 → savings 0.0% (correctness check: median IS optimal when over/understock cost equally)
- As margin rises, optimal order shifts toward higher quantiles → savings climb to 55.5%
- **Point forecasts leave the most money on the table exactly where margins are highest** — the core case for probabilistic forecasting

**Caveat:** assumed margins; the *method* and the *shape of the sensitivity curve* are the portfolio-worthy findings, both sound.

**Saved:** `src/inventory_optimizer.py`, `business_results.csv`, `sensitivity_analysis.csv`

In [26]:
# prep lightweight app data: forecasts + price + item metadata, top-N series by volume
lgb_fc = pd.read_parquet("data/processed/forecasts_lightgbm.parquet")
prices = pd.read_parquet("data/processed/features.parquet",
                         columns=["id","store_id","cat_id","dept_id","state_id","date","sell_price"])
last_price = prices.sort_values("date").groupby("id", observed=True).last().reset_index()

app_fc = lgb_fc.merge(
    last_price[["id","store_id","cat_id","dept_id","state_id","sell_price"]],
    on="id", how="left")
app_fc["sell_price"] = app_fc["sell_price"].fillna(app_fc["sell_price"].median())

# keep it light: top 500 series by total forecast volume (dashboard doesn't need all 30k)
top = (app_fc.groupby("id")["p50"].sum().sort_values(ascending=False).head(500).index)
app_fc = app_fc[app_fc["id"].isin(top)].reset_index(drop=True)
app_fc.to_parquet("app/data/app_forecasts.parquet", index=False)
print("app data:", app_fc.shape, "|", app_fc["id"].nunique(), "series")

/tmp/ipykernel_6128/291066334.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top = (app_fc.groupby("id")["p50"].sum().sort_values(ascending=False).head(500).index)


app data: (14000, 11) | 500 series


In [27]:
import shutil, os
for f in ["evaluation_results.csv","business_results.csv","sensitivity_analysis.csv"]:
    shutil.copy(f"data/processed/{f}", f"app/data/{f}")
print("copied support files")

copied support files
